# Hand Detector từ HaGRID pretrained — không cần training

**Mục tiêu**: tải checkpoint **YOLOv10n_hands.pt** đã train sẵn trên HaGRIDv2 (1M+ ảnh, mAP@0.5 = 87.9% theo paper) → save vào Drive → dùng trực tiếp trong pipeline.

Nguồn chính thức: https://github.com/hukenovs/hagrid (Kapitanov et al., WACV 2024).

## Flow

1. Tải file `.pt` (~6 MB)
2. Sanity test trên 1 ảnh tay
3. Log model artifact lên W&B (tracking)
4. Copy về Drive

Tổng thời gian: < 5 phút.

## Khi báo cáo / demo

**Wording đề xuất** trong slide:
> *"For hand detection, we adopt the YOLOv10n model pretrained by Kapitanov et al. (2024) on the HaGRIDv2 dataset (1.5 TB, 1.08 million RGB images, 33 gesture classes). The pretrained model achieves 87.9% mAP@0.5 on the HaGRID test set and provides robust hand localisation across diverse poses and lighting."*

Cite paper: Kapitanov et al., "HaGRID — HAnd Gesture Recognition Image Dataset", WACV 2024. arXiv:2206.08219.

In [ ]:
# Setup
!pip install -q ultralytics wandb

import os, urllib.request, hashlib
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/signlang'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Output: {SAVE_DIR}')

## 1 · Download pretrained checkpoint

In [ ]:
URL = 'https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid_v2/models/YOLOv10n_hands.pt'
OUT = '/content/yolov10n_hands.pt'

if not os.path.exists(OUT):
    print(f'Downloading {URL}…')
    urllib.request.urlretrieve(URL, OUT)

size_mb = os.path.getsize(OUT) / 1024**2
with open(OUT, 'rb') as f:
    sha = hashlib.sha256(f.read()).hexdigest()[:12]
print(f'Downloaded: {OUT}\n  Size: {size_mb:.1f} MB\n  SHA-256 (first 12): {sha}')

## 2 · Sanity test trên 1 ảnh

Kiểm tra checkpoint load được + detect được tay. Upload 1 ảnh tay bất kỳ (hoặc dùng test image từ ASL Alphabet đã download trước đó nếu có).

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

model = YOLO(OUT)
print(f'Model loaded: {model.task}, device={model.device}, names={model.names}')

# Tìm 1 ảnh test sẵn có (từ Kaggle ASL test nếu đã download), hoặc cho user upload
import glob
candidates = (
    glob.glob('/content/data/asl_alphabet_test/asl_alphabet_test/*.jpg')
    + glob.glob('/content/hand_dataset/test/images/*.jpg')
    + glob.glob('/content/hand_dataset/valid/images/*.jpg')
)
if candidates:
    test_img = candidates[0]
    print(f'Using existing image: {test_img}')
else:
    from google.colab import files
    print('Upload 1 ảnh tay bất kỳ (jpg/png):')
    uploaded = files.upload()
    test_img = list(uploaded.keys())[0]

results = model.predict(source=test_img, conf=0.25, verbose=False)
annotated = results[0].plot()
plt.figure(figsize=(8, 6))
plt.imshow(annotated[..., ::-1]); plt.axis('off'); plt.title(f'{len(results[0].boxes)} hand(s) detected')
plt.show()

if len(results[0].boxes) == 0:
    print('⚠️  No hand detected. Thử 1 ảnh tay rõ ràng hơn.')
else:
    for i, box in enumerate(results[0].boxes):
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print(f'  Hand {i+1}: bbox=({x1:.0f},{y1:.0f})→({x2:.0f},{y2:.0f}) conf={conf:.3f}')

## 3 · Log artifact lên W&B (để tracking + có entry trong project)

In [ ]:
import wandb
wandb.login()

run = wandb.init(
    project='signlang-detector',
    name='hagrid-pretrained-yolov10n',
    config={
        'source': 'HaGRID-pretrained',
        'architecture': 'YOLOv10n',
        'pretrain_dataset': 'HaGRIDv2 (1.08M images, 1.5TB)',
        'reported_mAP50_on_hagrid': 0.879,
        'reference': 'Kapitanov et al., WACV 2024 — arXiv:2206.08219',
        'download_url': URL,
        'sha256_prefix': sha,
        'size_mb': size_mb,
    },
    tags=['detector', 'pretrained', 'hagrid', 'yolov10n'],
)

art = wandb.Artifact(
    name='yolov10n-hands-hagrid-pretrained', type='model',
    description='HaGRIDv2-pretrained YOLOv10n hand detector. mAP@0.5=0.879 (per paper).',
    metadata={'mAP50_paper': 0.879, 'pretrain_dataset': 'HaGRIDv2', 'arch': 'YOLOv10n'},
)
art.add_file(OUT, name='best.pt')
wandb.log_artifact(art)
wandb.summary['mAP50_reported'] = 0.879
wandb.finish()
print('✓ Logged to W&B.')

## 4 · Copy vào Drive cho pipeline

In [ ]:
import shutil
DEST = f'{SAVE_DIR}/yolov10n_hands.pt'
shutil.copy(OUT, DEST)
print(f'✓ Saved: {DEST}\n  Size: {os.path.getsize(DEST)/1024**2:.1f} MB')

## Bước tiếp — chạy demo trên máy local

1. Tải `yolov10n_hands.pt` từ Google Drive về máy.
2. Đặt vào path mà `configs/demo.yaml` của repo expect:
   ```bash
   cd CV/Project/signlang
   mkdir -p runs/detector/weights
   cp ~/Drive/MyDrive/signlang/yolov10n_hands.pt runs/detector/weights/best.pt
   cp ~/Drive/MyDrive/signlang/cnn_resnet18.pt runs/classifier_resnet18/best.pt
   ```
3. Chạy:
   ```bash
   make demo            # OpenCV window — cần webcam
   make demo-web        # hoặc Gradio @ localhost:7860
   ```

## Note: tương thích với code repo

Class `HandDetector` của repo dùng `ultralytics.YOLO(weights)` — tương thích với cả YOLOv8 và YOLOv10 (cùng API). Không cần sửa code.

## Bonus: nếu giảng viên hỏi "tại sao không tự train?"

Trả lời: *"HaGRIDv2 là dataset 1.5 TB / 1M+ ảnh được curate kỹ với 65k+ persons. Train từ đầu trên Colab free (Tesla T4, 16 GB VRAM) sẽ mất nhiều ngày và không tốt hơn pretrained do dataset chúng tôi access được nhỏ hơn ~100×. Chúng tôi chọn cách kế thừa pretrained — cùng triết lý transfer learning đã dùng cho classifier ResNet18 (ImageNet pretrained → fine-tune ASL)."*

Đây là **defense vững** vì show được hiểu trade-off compute / data.